# 05 — Playground : la boucle agent dans sa forme définitive

**Jalon 2, avant le code de `agent/boucle.py` et `cli.py`.** Le notebook 04 a montré ce qu'est
une boucle d'agent. Celui-ci essaie la **forme décidée au cadrage**, avant de l'écrire dans le
package :

1. la boucle manuelle avec ses **dépendances injectées** (client MCP, client Anthropic), un
   **statut** de fin et **une ligne de log JSON par tour** d'API ;
2. le **faux client Anthropic** : faire tourner la boucle sans API, comme le feront les tests ;
3. l'**auto-correction** après une erreur SQL, provoquée exprès ;
4. les **arrêts anormaux** forcés (`max_tours`, `max_tokens`) ;
5. **Haiku 5.5 contre Sonnet 5.5**.

Les parties 1 et 2 sont **gratuites** : aucun appel à l'API, la clé n'est même pas lue.
Les parties 3 à 5 appellent l'API ; les tokens consommés sont affichés à chaque fois.

**Prérequis :** base démarrée (`docker compose up -d`), `ANTHROPIC_API_KEY` dans le `.env`
(parties 3 à 5), kernel `.venv`.

## 0. Préparation

Les logs de l'agent passent par le même `configure_logging` que le serveur : une ligne JSON par
événement, sur stderr. Dans Jupyter, stderr s'affiche sous la cellule (sur fond coloré).

In [ ]:
import os
from pathlib import Path

# Le .env est à la racine du repo : on s'y place (le serveur MCP le lira aussi).
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

import json
import logging
import sys
import time
from typing import Any, Literal

from mcp import Client, StdioServerParameters
from mcp.types import Implementation
from pydantic import BaseModel, JsonValue

from text_to_sql_mcp.logging_config import LOGGER_NAME, configure_logging

configure_logging("INFO")
logger = logging.getLogger(f"{LOGGER_NAME}.agent")

SONNET = "claude-sonnet-5-5"
HAIKU = "claude-haiku-5-5"
# Le serveur est lancé avec le même interpréteur que le notebook : pas de dépendance au PATH.
SERVEUR = StdioServerParameters(command=sys.executable, args=["-m", "text_to_sql_mcp.server"])

In [ ]:
PROMPT_V0 = """\
Tu es un analyste de données. Tu réponds à des questions métier en interrogeant une base
PostgreSQL avec les outils fournis.
- Commence par list_tables, puis get_schema sur les tables utiles. N'invente jamais de table
  ni de colonne.
- Si la question ne peut pas être résolue avec les tables disponibles, dis-le clairement.
- Réponds en français, en une ou deux phrases, puis affiche le SQL exécuté (champ
  executed_sql du résultat de run_query) dans un bloc ```sql.
"""

## 1. La boucle, dans sa forme définitive

Trois différences avec le notebook 04.

**Les dépendances sont injectées.** `run_agent` reçoit un client MCP **déjà ouvert** et un
client Anthropic. Elle ne sait pas si le serveur tourne en sous-processus ou en mémoire, ni si
le client Anthropic est vrai ou faux. C'est ce qui la rend testable (partie 2). Une petite
enveloppe, `ask`, ouvre le vrai serveur en stdio : c'est elle qu'appellera le CLI.

**La fin est un statut, pas une exception.** Un agent qui tourne en rond est une mesure pour
l'évaluation, pas un plantage :

| `stop_reason` de l'API | statut de l'agent |
|---|---|
| `end_turn` | `ok` |
| `tool_use` au dernier tour autorisé | `max_tours` |
| `max_tokens` | `max_tokens` |
| `refusal` | `refus_modele` |
| tout autre (`pause_turn`, `model_context_window_exceeded`…) | `arret_inattendu` |

Seules les pannes d'infrastructure (API injoignable, serveur MCP qui plante) lèvent une exception.

**Une ligne de log par tour** : client, numéro de tour, `stop_reason`, tokens, durée. Le champ
`client` est le même que dans les logs du serveur : on suit une question d'un flux à l'autre.

Un détail : si le dernier tour autorisé demande encore des outils, on **ne les exécute pas**.
Leurs résultats ne seraient jamais lus par le modèle.

In [ ]:
Statut = Literal["ok", "max_tours", "max_tokens", "refus_modele", "arret_inattendu"]

# stop_reason de l'API -> statut de l'agent. "tool_use" n'y figure pas : la boucle continue.
STATUT_PAR_STOP_REASON: dict[str, Statut] = {
    "end_turn": "ok",
    "max_tokens": "max_tokens",
    "refusal": "refus_modele",
}


class ToolCall(BaseModel):
    """Un appel d'outil observé par notre code."""

    name: str
    arguments: dict[str, Any]
    is_error: bool
    duration_ms: float
    structured: dict[str, Any] | None = None
    error: str | None = None


class AgentResult(BaseModel):
    """Ce que l'agent a fait, construit à partir de nos observations (jamais du texte du modèle)."""

    question: str
    model: str
    prompt: str
    client_name: str
    status: Statut
    stop_reason: str | None
    answer: str
    executed_sql: str | None
    columns: list[str] | None
    rows: list[list[JsonValue]] | None
    truncated: bool | None
    tool_calls: list[ToolCall]
    tool_errors: int
    turns: int
    input_tokens: int
    output_tokens: int
    duration_s: float

In [ ]:
async def executer_outil(client_mcp: Client, bloc: Any) -> tuple[ToolCall, dict[str, Any]]:
    """Exécute un bloc tool_use sur le serveur MCP : renvoie l'appel observé et le tool_result."""
    debut = time.perf_counter()
    resultat = await client_mcp.call_tool(bloc.name, bloc.input)
    texte = "\n".join(c.text for c in resultat.content if c.type == "text")
    appel = ToolCall(
        name=bloc.name,
        arguments=bloc.input,
        is_error=resultat.is_error,
        duration_ms=round((time.perf_counter() - debut) * 1000, 1),
        structured=resultat.structured_content,
        error=texte if resultat.is_error else None,
    )
    tool_result = {
        "type": "tool_result",
        "tool_use_id": bloc.id,  # relie le résultat à la demande de Claude
        "content": texte,
        "is_error": resultat.is_error,  # Claude sait qu'il doit corriger
    }
    return appel, tool_result


async def run_agent(
    question: str,
    *,
    client_mcp: Client,
    anthropic: Any,
    model: str,
    prompt_name: str,
    prompt: str,
    client_name: str,
    max_turns: int = 10,
    max_tokens: int = 8000,
) -> AgentResult:
    """La boucle tool use, écrite à la main, avec ses dépendances injectées."""
    debut = time.perf_counter()
    systeme = prompt + "\nInformations sur la base :\n" + (client_mcp.instructions or "")
    outils = [
        {"name": o.name, "description": o.description, "input_schema": o.input_schema}
        for o in (await client_mcp.list_tools()).tools
    ]
    messages: list[dict[str, Any]] = [{"role": "user", "content": question}]
    journal: list[ToolCall] = []
    usages = []
    reponse = None
    statut: Statut = "max_tours"  # valeur si la boucle épuise ses tours
    for tour in range(1, max_turns + 1):
        debut_tour = time.perf_counter()
        reponse = await anthropic.messages.create(
            model=model, max_tokens=max_tokens, system=systeme, tools=outils, messages=messages
        )
        usages.append(reponse.usage)
        logger.info(
            "tour_api",
            extra={
                "client": client_name,
                "tour": tour,
                "stop_reason": reponse.stop_reason,
                "tokens_entree": reponse.usage.input_tokens,
                "tokens_sortie": reponse.usage.output_tokens,
                "duree_ms": round((time.perf_counter() - debut_tour) * 1000, 1),
            },
        )
        # Tel quel, blocs thinking compris : l'API exige qu'ils reviennent inchangés.
        messages.append({"role": "assistant", "content": reponse.content})
        if reponse.stop_reason != "tool_use":
            statut = STATUT_PAR_STOP_REASON.get(reponse.stop_reason, "arret_inattendu")
            break
        if tour == max_turns:
            break  # plafond atteint : outils demandés non exécutés, statut max_tours
        resultats = []
        for bloc in reponse.content:
            if bloc.type == "tool_use":
                appel, tool_result = await executer_outil(client_mcp, bloc)
                journal.append(appel)
                resultats.append(tool_result)
        messages.append({"role": "user", "content": resultats})

    # Dernière requête réussie : c'est son résultat que l'évaluation comparera.
    reussies = [c for c in journal if c.name == "run_query" and not c.is_error]
    derniere = reussies[-1] if reussies else None
    donnees = (derniere.structured or {}) if derniere else {}
    return AgentResult(
        question=question,
        model=model,
        prompt=prompt_name,
        client_name=client_name,
        status=statut,
        stop_reason=reponse.stop_reason if reponse else None,
        answer="\n".join(b.text for b in reponse.content if b.type == "text") if reponse else "",
        executed_sql=donnees.get("executed_sql"),
        columns=donnees.get("columns"),
        rows=donnees.get("rows"),
        truncated=donnees.get("truncated"),
        tool_calls=journal,
        tool_errors=sum(c.is_error for c in journal),
        turns=len(usages),
        input_tokens=sum(u.input_tokens for u in usages),
        output_tokens=sum(u.output_tokens for u in usages),
        duration_s=round(time.perf_counter() - debut, 2),
    )


async def ask(
    question: str,
    *,
    anthropic: Any,
    model: str = SONNET,
    prompt_name: str = "v0",
    prompt: str = PROMPT_V0,
    max_turns: int = 10,
    max_tokens: int = 8000,
) -> AgentResult:
    """Enveloppe de production : lance le vrai serveur MCP en stdio, puis la boucle."""
    infos = Implementation(name=f"text-to-sql-agent:{model}:{prompt_name}", version="0.1.0")
    async with Client(SERVEUR, client_info=infos) as client_mcp:
        return await run_agent(
            question,
            client_mcp=client_mcp,
            anthropic=anthropic,
            model=model,
            prompt_name=prompt_name,
            prompt=prompt,
            client_name=f"{infos.name}/{infos.version}",  # même format que le serveur
            max_turns=max_turns,
            max_tokens=max_tokens,
        )

## 2. Le faux client Anthropic : la boucle sans API

`run_agent` n'a besoin que d'un objet qui possède `messages.create(...)`. Le faux client
**rejoue des réponses écrites à l'avance** et **enregistre chaque requête reçue**. Côté MCP, on
prend le client **en mémoire** `Client(mcp)` : vrai serveur, vraie base, mais pas de
sous-processus.

On teste ainsi **notre code**, pas le modèle : la boucle construit-elle le bon historique ?
relie-t-elle chaque résultat à sa demande ? le statut et l'`AgentResult` sont-ils justes ?
La qualité du SQL écrit par Claude, elle, sera mesurée au jalon 3.

In [ ]:
from anthropic.types import Message

from text_to_sql_mcp.server import mcp as serveur_mcp


def message(*blocs: dict[str, Any], stop_reason: str) -> Message:
    """Une réponse de l'API, construite à la main (même type que la vraie)."""
    return Message.model_validate(
        {
            "id": "msg_faux",
            "type": "message",
            "role": "assistant",
            "model": "faux",
            "content": list(blocs),
            "stop_reason": stop_reason,
            "stop_sequence": None,
            "usage": {"input_tokens": 100, "output_tokens": 10},
        }
    )


def outil(nom: str, id_: str, **arguments: Any) -> dict[str, Any]:
    return {"type": "tool_use", "id": id_, "name": nom, "input": arguments}


def texte(contenu: str) -> dict[str, Any]:
    return {"type": "text", "text": contenu}


class _FauxMessages:
    def __init__(self, reponses: list[Message]) -> None:
        self._reponses = list(reponses)
        self.requetes: list[dict[str, Any]] = []

    async def create(self, **requete: Any) -> Message:
        # Copie de la liste : la boucle la modifie ensuite, on veut l'état au moment de l'appel.
        self.requetes.append({**requete, "messages": list(requete["messages"])})
        return self._reponses.pop(0)


class FauxAnthropic:
    """Rejoue des réponses écrites à l'avance et enregistre les requêtes reçues."""

    def __init__(self, reponses: list[Message]) -> None:
        self.messages = _FauxMessages(reponses)


async def jouer(scenario: list[Message], max_turns: int = 10) -> tuple[AgentResult, FauxAnthropic]:
    """Fait tourner la boucle sur un scénario, avec le serveur MCP en mémoire."""
    faux = FauxAnthropic(scenario)
    infos = Implementation(name="faux-agent", version="0")
    async with Client(serveur_mcp, client_info=infos) as client_mcp:
        r = await run_agent(
            "question de test",
            client_mcp=client_mcp,
            anthropic=faux,
            model="faux",
            prompt_name="v0",
            prompt=PROMPT_V0,
            client_name="faux-agent/0",
            max_turns=max_turns,
        )
    return r, faux

### 2.1 Scénario nominal

Le faux Claude demande `list_tables`, puis une requête, puis répond. Les outils, eux, sont
**vraiment exécutés** sur la base.

Le serveur tourne ici dans le même processus : ses lignes `appel_outil` s'affichent sur stderr,
entrelacées avec les lignes `tour_api` de la boucle, avec **le même champ `client`**.

In [ ]:
r, faux = await jouer(
    [
        message(outil("list_tables", "t1"), stop_reason="tool_use"),
        message(
            outil("run_query", "t2", sql="SELECT name FROM genre ORDER BY name LIMIT 3"),
            stop_reason="tool_use",
        ),
        message(texte("Alternative, Alternative & Punk, Blues."), stop_reason="end_turn"),
    ]
)
print(
    r.model_dump_json(indent=2, include={"status", "turns", "executed_sql", "rows", "tool_errors"})
)

Ce que la boucle a envoyé au **2ᵉ tour** : le dernier message est un message `user` qui contient
le `tool_result`, relié par `tool_use_id` à la demande `t1`. C'est exactement ce qu'un test
vérifiera avec un `assert`.

In [ ]:
dernier = faux.messages.requetes[1]["messages"][-1]
print("rôle :", dernier["role"])
for bloc in dernier["content"]:
    print(
        bloc["type"],
        bloc["tool_use_id"],
        "is_error =",
        bloc["is_error"],
        repr(bloc["content"][:50]),
    )

### 2.2 Erreur SQL puis correction

Le faux Claude se trompe de colonne (`nom` au lieu de `name`), reçoit l'erreur de PostgreSQL,
puis corrige. L'erreur est **tracée** (`tool_errors`), et le SQL retenu est celui de la
**dernière requête réussie**.

In [ ]:
r, faux = await jouer(
    [
        message(outil("run_query", "t1", sql="SELECT nom FROM genre"), stop_reason="tool_use"),
        message(
            outil("run_query", "t2", sql="SELECT name FROM genre LIMIT 2"), stop_reason="tool_use"
        ),
        message(texte("Rock et Jazz."), stop_reason="end_turn"),
    ]
)
print("statut :", r.status, "| erreurs d'outils :", r.tool_errors, "| SQL retenu :", r.executed_sql)
print("ce que Claude a lu :", r.tool_calls[0].error)

### 2.3 Arrêts anormaux

Un faux Claude qui demande `list_tables` en boucle, plafonné à 2 tours ; puis une réponse coupée
par `max_tokens`.

In [ ]:
en_boucle = [message(outil("list_tables", f"t{i}"), stop_reason="tool_use") for i in range(5)]
r, _ = await jouer(en_boucle, max_turns=2)
print("plafond   :", r.status, "| tours :", r.turns, "| outils exécutés :", len(r.tool_calls))

r, _ = await jouer([message(texte("Les 5 artistes sont"), stop_reason="max_tokens")])
print("tronquée  :", r.status, "| réponse :", repr(r.answer))

r, _ = await jouer([message(texte("…"), stop_reason="pause_turn")])
print("imprévu   :", r.status, "| stop_reason :", r.stop_reason)

Au 2ᵉ tour du premier cas, Claude demandait encore `list_tables` : la boucle ne l'a **pas
exécuté** (1 seul outil exécuté pour 2 tours).

Ces scénarios deviendront `tests/test_agent.py`, avec des `assert` à la place des `print`.
Ils tournent en une fraction de seconde et ne coûtent rien.

## 3. Avec la vraie API : auto-correction provoquée

À partir d'ici, chaque cellule consomme des tokens. On lit la clé, puis on définit un affichage
compact du résultat.

In [ ]:
from anthropic import AsyncAnthropic
from pydantic import SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict


class CleAPI(BaseSettings):
    """Lit uniquement la clé Anthropic dans le .env."""

    model_config = SettingsConfigDict(env_file=".env", extra="ignore")
    anthropic_api_key: SecretStr


anthropic = AsyncAnthropic(api_key=CleAPI().anthropic_api_key.get_secret_value())


def resume(r: AgentResult) -> None:
    """Une vue compacte d'un AgentResult."""
    print(
        f"[{r.status}] {r.model} | tours={r.turns} entrée={r.input_tokens} "
        f"sortie={r.output_tokens} durée={r.duration_s}s erreurs d'outils={r.tool_errors}"
    )
    for c in r.tool_calls:
        arguments = json.dumps(c.arguments, ensure_ascii=False)
        print(f"   {c.name}({arguments[:90]}) -> {'ERREUR' if c.is_error else 'ok'}")
    print("   SQL retenu :", r.executed_sql)

Pour **forcer** une erreur, une variante de prompt **volontairement fausse** : elle interdit
`get_schema` et affirme une convention de nommage qui n'est pas celle de la base. Elle ne sert
qu'ici.

Deux issues possibles, toutes deux instructives :
- Claude écrit `"BillingCountry"`, reçoit l'erreur `42703` (colonne inconnue) et se corrige :
  regarde `tool_errors` et la suite des appels ;
- Claude repère la contradiction avec les instructions du serveur (snake_case) et ne se trompe
  pas, si besoin on passe à Sonnet.

In [ ]:
PROMPT_PIEGE = (
    PROMPT_V0
    + """\
- N'appelle jamais get_schema : la base suit la convention PascalCase de Chinook, avec des
  identifiants entre guillemets ("Invoice"."BillingCountry", "Total"…).
"""
)

r = await ask(
    "Quel est le chiffre d'affaires total par pays de facturation ? Donne les 5 premiers.",
    anthropic=anthropic,
    model=HAIKU,
    prompt_name="piege",
    prompt=PROMPT_PIEGE,
)
resume(r)
for c in r.tool_calls:
    if c.is_error:
        print("\nErreur lue par Claude :", c.error)
print("\n" + r.answer)

On constate que même Haiku arrive à se corriger et à répondre (il détecte quand même le snake_case avec le retour de la base)

## 4. Arrêts forcés sur la vraie API

Les mêmes arrêts que dans la partie 2.3, mais provoqués par de vrais réglages. Haiku 5.5, pour
le coût.

In [ ]:
QUESTION_CA = "Quel est le chiffre d'affaires total par pays ?"

r = await ask(QUESTION_CA, anthropic=anthropic, model=HAIKU, max_turns=2)
resume(r)  # attendu : max_tours (il faut au moins 3 tours pour répondre)

r = await ask(QUESTION_CA, anthropic=anthropic, model=HAIKU, max_tokens=16)
resume(r)  # attendu : max_tokens (16 tokens ne suffisent même pas pour un appel d'outil)

## 5. Haiku 5.5 contre Sonnet 5.5

Une question de référence et une question sans réponse. On compare le statut, les tours, les
tokens, la durée, et le **résultat** des requêtes (execution accuracy en miniature : les lignes,
sans l'ordre ni les noms de colonnes).

In [ ]:
QUESTIONS = [
    "Quels sont les 5 artistes qui ont publié le plus d'albums ?",
    "Quel est le salaire moyen des employés ?",
]
resultats = {}
for question in QUESTIONS:
    for modele in (HAIKU, SONNET):
        resultats[question, modele] = await ask(question, anthropic=anthropic, model=modele)

for question in QUESTIONS:
    print(f"=== {question}")
    for modele in (HAIKU, SONNET):
        resume(resultats[question, modele])
    lignes = [sorted(map(str, resultats[question, m].rows or [])) for m in (HAIKU, SONNET)]
    print("Même résultat (ordre ignoré) :", lignes[0] == lignes[1], "\n")

Sur la question « salaire », le bon comportement est **aucune requête** (ou une requête
d'exploration) et une réponse qui dit que la donnée n'existe pas. `executed_sql` vide n'y est pas
un échec. Le harnais du jalon 3 devra traiter ce cas à part.

Lis aussi les lignes `tour_api` affichées sur stderr : une par tour, avec le même champ `client`
que les lignes `appel_outil` du serveur (visibles dans le terminal qui a lancé Jupyter).

## 6. À retenir, et ce que le code reprendra

- `run_agent` est la boucle pure, avec ses dépendances injectées ; `ask` l'enveloppe stdio.
  Le CLI appellera `ask`, les tests appelleront `run_agent` avec le faux client.
- Fin de boucle = un **statut** ; seules les pannes d'infrastructure lèvent une exception.
- Les erreurs SQL reviennent à Claude avec `is_error: true` : il se corrige seul, et
  `tool_errors` le trace.
- Le SQL retenu est celui de la dernière requête **réussie**.

Différences avec le package :

| Ici | Dans `src/text_to_sql_mcp/` |
|---|---|
| `PROMPT_V0` dans une cellule | `agent/prompts/v0.md`, chargé par son nom |
| `CleAPI` | `AgentSettings` dans `config.py` (clé, modèle, prompt, tours, tokens) |
| `print` / `resume` | CLI `text-to-sql-agent`, sortie texte ou `--json` |
| scénarios de la partie 2 | `tests/test_agent.py` |

:)